In [3]:
from __future__ import annotations

import os
import csv
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


@dataclass(frozen=True)
class ResultRow:
    cv_id: int
    seed: int
    val_metric: str
    val_score: float
    test_acc: float
    test_macro_f1: float
    test_top5: float
    best_trial: int
    timestamp: str
    run_dir: str  # cv-*_seed-* directory name


def _safe_float(x: Any) -> float:
    try:
        return float(x)
    except Exception:
        return float("nan")


def _safe_int(x: Any) -> int:
    try:
        return int(float(x))
    except Exception:
        return -1


def _mean(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]  # drop NaN
    return sum(xs) / len(xs) if xs else float("nan")


def _std(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    if len(xs) <= 1:
        return float("nan")
    m = _mean(xs)
    v = sum((x - m) ** 2 for x in xs) / (len(xs) - 1)
    return v ** 0.5


def _min(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    return min(xs) if xs else float("nan")


def _max(xs: List[float]) -> float:
    xs = [x for x in xs if x == x]
    return max(xs) if xs else float("nan")


def _read_metrics_csv(metrics_path: Path) -> Optional[Dict[str, Any]]:
    if not metrics_path.exists():
        return None
    with metrics_path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    if not rows:
        return None
    return rows[0]


def _parse_cv_seed_from_dirname(name: str) -> Tuple[int, int]:
    # expected: cv-0_seed-12
    m = re.match(r"cv-(\d+)_seed-(\d+)", name)
    if not m:
        return -1, -1
    return int(m.group(1)), int(m.group(2))


def load_results(save_root: str | Path, include_best_params: bool = False) -> Tuple[List[ResultRow], Optional[List[Dict[str, Any]]]]:
    """
    save_root/
      cv-0_seed-0/metrics.csv
      cv-0_seed-0/best_param.json (optional)
      ...

    Returns:
      rows: list of ResultRow
      best_params_list: list of dict (if include_best_params=True else None)
    """
    save_root = Path(save_root)
    if not save_root.exists():
        raise FileNotFoundError(f"save_root not found: {save_root}")

    rows: List[ResultRow] = []
    best_params_list: List[Dict[str, Any]] = []

    subdirs = sorted([p for p in save_root.iterdir() if p.is_dir()])

    for d in subdirs:
        cv_id, seed = _parse_cv_seed_from_dirname(d.name)
        metrics_path = d / "metrics.csv"
        r = _read_metrics_csv(metrics_path)
        if r is None:
            # metrics.csv が無い/壊れている場合はスキップ
            continue

        row = ResultRow(
            cv_id=cv_id if cv_id != -1 else _safe_int(r.get("cv_id", -1)),
            seed=seed if seed != -1 else _safe_int(r.get("seed", -1)),
            val_metric=str(r.get("val_metric", "")),
            val_score=_safe_float(r.get("val_score", "nan")),
            test_acc=_safe_float(r.get("test_acc", "nan")),
            test_macro_f1=_safe_float(r.get("test_macro_f1", "nan")),
            test_top5=_safe_float(r.get("test_top5", "nan")),
            best_trial=_safe_int(r.get("best_trial", -1)),
            timestamp=str(r.get("timestamp", "")),
            run_dir=d.name,
        )
        rows.append(row)

        if include_best_params:
            bp = d / "best_param.json"
            if bp.exists():
                try:
                    best_params_list.append(json.loads(bp.read_text(encoding="utf-8")))
                except Exception:
                    best_params_list.append({"__error__": f"failed to read {bp}"})

    return rows, (best_params_list if include_best_params else None)


def print_results(save_root: str | Path, include_best_params: bool = False) -> None:
    """
    保存先を入力すると結果を表示する関数である。
    """
    rows, best_params_list = load_results(save_root, include_best_params=include_best_params)
    if not rows:
        print(f"[WARN] No results found under: {Path(save_root).resolve()}", flush=True)
        return

    # 全体集計
    accs = [r.test_acc for r in rows]
    f1s = [r.test_macro_f1 for r in rows]
    top5s = [r.test_top5 for r in rows]
    vals = [r.val_score for r in rows]

    print("=" * 72, flush=True)
    print(f"[RESULTS] root = {Path(save_root).resolve()}", flush=True)
    print(f"[RESULTS] runs = {len(rows)} (cv x seed)", flush=True)
    print("-" * 72, flush=True)
    print(
        "TEST  acc     : mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(accs), _std(accs), _min(accs), _max(accs)
        ),
        flush=True,
    )
    print(
        "TEST  macro_f1: mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(f1s), _std(f1s), _min(f1s), _max(f1s)
        ),
        flush=True,
    )
    print(
        "TEST  top5    : mean={:.4f} std={:.4f} min={:.4f} max={:.4f}".format(
            _mean(top5s), _std(top5s), _min(top5s), _max(top5s)
        ),
        flush=True,
    )
    print(
        "VAL({}) score : mean={:.4f} std={:.4f}".format(
            rows[0].val_metric, _mean(vals), _std(vals)
        ),
        flush=True,
    )

    # foldごとの平均（seed平均）
    by_cv: Dict[int, List[ResultRow]] = {}
    for r in rows:
        by_cv.setdefault(r.cv_id, []).append(r)

    print("-" * 72, flush=True)
    print("[BY CV] (seed average per fold)", flush=True)
    for cv_id in sorted(by_cv.keys()):
        rs = by_cv[cv_id]
        a = _mean([x.test_acc for x in rs])
        f = _mean([x.test_macro_f1 for x in rs])
        t = _mean([x.test_top5 for x in rs])
        print(f"  cv={cv_id:2d} | n={len(rs):2d} | acc={a:.4f}  f1={f:.4f}  top5={t:.4f}", flush=True)

    # 代表としてトップ数件の個別結果
    print("-" * 72, flush=True)
    print("[TOP RUNS] by test_acc (top 10)", flush=True)
    rows_sorted = sorted(rows, key=lambda r: (r.test_acc if r.test_acc == r.test_acc else -1.0), reverse=True)
    for i, r in enumerate(rows_sorted[:10]):
        print(
            f"  #{i+1:2d} {r.run_dir:14s} | acc={r.test_acc:.4f} f1={r.test_macro_f1:.4f} top5={r.test_top5:.4f} | val={r.val_score:.4f}",
            flush=True,
        )

    if include_best_params and best_params_list is not None:
        # best paramを軽く眺める（完全表示だと長くなるので、上位の試行だけ表示）
        print("-" * 72, flush=True)
        print("[BEST PARAMS] (first 3 entries)", flush=True)
        for i, bp in enumerate(best_params_list[:3]):
            print(f"  --- {i+1} ---", flush=True)
            # よく見る所だけ出す
            trial_params = bp.get("best_trial_params", {})
            resolved = bp.get("resolved_hyperparams", {})
            print(f"  best_value: {bp.get('best_value')}", flush=True)
            print(f"  best_trial_params: {trial_params}", flush=True)
            print(f"  resolved_hyperparams: {resolved}", flush=True)

    print("=" * 72, flush=True)

In [4]:
ROOT_PATH = "/home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results"

print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_cifar_10_esn_units-512"), include_best_params=True)
print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_cifar_10_bi_esn_units-512"), include_best_params=True)
print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_cifar_10_bi_esn2d_units-512"), include_best_params=True)
print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_mnist_esn_units-512"), include_best_params=True)
print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_mnist_bi_esn_units-512"), include_best_params=True)
print_results(os.path.join(ROOT_PATH, "2026-02-01_00-19-29_mnist_bi_esn2d_units-512"), include_best_params=True)

[RESULTS] root = /home/nakanishi/WORKSPACE/bidirectional_2D_reservoir_computing/results/2026-02-01_00-19-29_cifar_10_esn_units-512
[RESULTS] runs = 15 (cv x seed)
------------------------------------------------------------------------
TEST  acc     : mean=0.2121 std=0.0582 min=0.1170 max=0.3063
TEST  macro_f1: mean=0.2009 std=0.0663 min=0.0691 max=0.3047
TEST  top5    : mean=0.6734 std=0.0692 min=0.5271 max=0.7885
VAL(acc) score : mean=0.2863 std=0.0196
------------------------------------------------------------------------
[BY CV] (seed average per fold)
  cv= 0 | n= 3 | acc=0.1863  f1=0.1719  top5=0.6385
  cv= 1 | n= 3 | acc=0.2099  f1=0.1993  top5=0.6737
  cv= 2 | n= 3 | acc=0.2348  f1=0.2274  top5=0.7067
  cv= 3 | n= 3 | acc=0.2101  f1=0.1917  top5=0.6520
  cv= 4 | n= 3 | acc=0.2195  f1=0.2143  top5=0.6960
------------------------------------------------------------------------
[TOP RUNS] by test_acc (top 10)
  # 1 cv-4_seed-0    | acc=0.3063 f1=0.3047 top5=0.7885 | val=0.2886
  